# `find_plane_normal()`

`nematics3d.find_plane_normal()` fits a least-squares plane to a finite three-dimensional point cloud. It returns a structured `PlaneNormalResult` containing the fitted unit normal, the point-cloud centroid, and diagnostics describing how planar the cloud is and how well the normal direction is determined.

The fitted plane passes through the centroid. Its normal is the direction along which the centered points have the smallest total squared extent.

**The sign of the returned normal is intrinsically ambiguous.** A plane with normal $\mathbf{n}$ is the same geometric plane as one with normal $-\mathbf{n}$.


## Setup

**For readers who are only interested in the tutorial, this section can be safely skipped.** Run the following cell to import `NumPy` and `Nematics3D`; no setup detail is needed to understand the examples.


In [1]:
import numpy as np
import nematics3d as n3d

## Minimal example: fit an exact plane

Consider points sampled from

$$
z = 0.5x - 0.25y + 2.
$$

An equivalent plane equation is

$$
0.5x - 0.25y - z + 2 = 0,
$$

so one valid normal is proportional to

$$
\mathbf{n}_\mathrm{expected}=(0.5,-0.25,-1).
$$


In [2]:
x = np.array([-2.0, -1.0, 0.0, 1.0, 2.0, 0.5])
y = np.array([1.0, -2.0, 0.5, 2.0, -1.0, -1.5])
z = 0.5 * x - 0.25 * y + 2.0
points = np.column_stack([x, y, z])

result = n3d.find_plane_normal(points)

expected_normal = np.array([0.5, -0.25, -1.0])
expected_normal /= np.linalg.norm(expected_normal)

print("normal:", result.normal)
print("centroid:", result.centroid)
print("axis overlap:", abs(np.dot(result.normal, expected_normal)))
print("planarity score:", result.planarity_score)
print("RMS thickness:", result.thickness_rms)

normal: [-0.43643578  0.21821789  0.87287156]
centroid: [ 0.08333333 -0.16666667  2.08333333]
axis overlap: 0.9999999999999999
planarity score: 1.0
RMS thickness: 0.0


The comparison uses $|\mathbf{n}_1\cdot\mathbf{n}_2|$ because the sign of an eigenvector is arbitrary. For an exact plane, the overlap should be 1 up to floating-point roundoff, `planarity_score` should be close to 1, and `thickness_rms` should be close to 0.


## Inputs and outputs

The public signature is:

```python
find_plane_normal(points)
```

### Input

`points` must be a finite three-dimensional point collection with shape `(N, 3)` and at least three rows. Integer and floating-point coordinates are accepted. Non-finite values and inputs with a coordinate dimension other than three are rejected.

The function does not require the points to define a unique plane. Collinear or coincident point clouds are accepted, but their fitted normal is geometrically underdetermined; the diagnostics below are intended to reveal that situation.

### Returned result

The function returns a `PlaneNormalResult`, which is a `ResultBase`, with these fields:

| Field | Meaning |
| --- | --- |
| `normal` | Unit normal of the least-squares best-fit plane. Its sign is arbitrary. |
| `centroid` | Mean of the input coordinates; the fitted plane passes through this point. |
| `planarity_score` | Dimensionless score in $[0,1]$ measuring how small the variance normal to the fitted plane is relative to the total variance. |
| `thickness_rms` | RMS point-cloud thickness along the fitted normal, in the same length units as the input coordinates. |
| `linearity_risk` | Ratio of the two smallest second-moment eigenvalues; values near 1 indicate that the normal direction is poorly determined because the cloud is close to one-dimensional. |
| `eigenvalues` | Three ascending eigenvalues of the centered point-cloud second-moment matrix. |

`result.metric` provides the five diagnostic fields other than `normal` as a shallow dictionary. This is useful when another result object needs to carry the plane-fit diagnostics without discarding the structured result interface.


In [3]:
print("normal:", result.normal)
print("centroid:", result.centroid)
print("planarity score:", result.planarity_score)
print("RMS thickness:", result.thickness_rms)
print("linearity risk:", result.linearity_risk)
print("eigenvalues:", result.eigenvalues)
print("metric keys:", tuple(result.metric))

normal: [-0.43643578  0.21821789  0.87287156]
centroid: [ 0.08333333 -0.16666667  2.08333333]
planarity score: 1.0
RMS thickness: 0.0
linearity risk: 0.0
eigenvalues: [ 0.         10.83791252 15.19333748]
metric keys: ('centroid', 'planarity_score', 'thickness_rms', 'eigenvalues', 'linearity_risk')


## Details

Let the input points be $\mathbf{x}_i$ and their centroid be

$$
\bar{\mathbf{x}}=\frac{1}{N}\sum_i \mathbf{x}_i.
$$

Define centered coordinates

$$
\mathbf{r}_i=\mathbf{x}_i-\bar{\mathbf{x}},
$$

and the second-moment matrix

$$
M=\sum_i \mathbf{r}_i\mathbf{r}_i^T.
$$

For any unit vector $\mathbf{n}$,

$$
\mathbf{n}^TM\mathbf{n}
=
\sum_i (\mathbf{r}_i\cdot\mathbf{n})^2.
$$

Therefore the eigenvector belonging to the smallest eigenvalue $\lambda_0$ minimizes the summed squared perpendicular distances to a plane through the centroid. `find_plane_normal()` uses this eigenvector as the fitted normal.

With ascending eigenvalues

$$
\lambda_0\leq\lambda_1\leq\lambda_2,
$$

the diagnostics are

$$
\text{planarity\_score}
=
\operatorname{clip}\!\left(
1-\frac{3\lambda_0}{\lambda_0+\lambda_1+\lambda_2},
0,1
\right),
$$

$$
\text{thickness\_rms}
=
\sqrt{\frac{\lambda_0}{N}},
$$

and, when $\lambda_1>0$,

$$
\text{linearity\_risk}
=
\frac{\lambda_0}{\lambda_1}.
$$

For an exactly one-dimensional cloud, $\lambda_0=\lambda_1=0$ and `linearity_risk` is defined as 1.


## Noisy plane

For a cloud with finite thickness, the normal remains a least-squares estimate rather than an exact constraint. The next example adds reproducible Gaussian noise only along $z$.


In [4]:
rng = np.random.default_rng(7)

x_noisy = rng.uniform(-2.0, 2.0, 200)
y_noisy = rng.uniform(-2.0, 2.0, 200)
z_noisy = (
    0.5 * x_noisy
    - 0.25 * y_noisy
    + 2.0
    + rng.normal(scale=0.05, size=200)
)
noisy_points = np.column_stack([x_noisy, y_noisy, z_noisy])

noisy_result = n3d.find_plane_normal(noisy_points)

print("axis overlap:", abs(np.dot(noisy_result.normal, expected_normal)))
print("planarity score:", noisy_result.planarity_score)
print("RMS thickness:", noisy_result.thickness_rms)
print("linearity risk:", noisy_result.linearity_risk)

axis overlap: 0.9999920886812601
planarity score: 0.99848849483874
RMS thickness: 0.03995848629486977
linearity risk: 0.0011973251977143995


The fitted axis should remain close to the exact normal, while `thickness_rms` becomes nonzero. Unlike `planarity_score`, `thickness_rms` carries the physical length scale of the data.


## Why `linearity_risk` matters

A perfectly straight line lies in infinitely many planes. Such a point cloud can have `planarity_score == 1` because there is zero variance along at least one normal direction, even though no unique plane normal exists.

For this reason, do not interpret `planarity_score` alone as a confidence score for the direction of `normal`. Use it together with `linearity_risk`.


In [5]:
line_points = np.column_stack([
    np.linspace(-2.0, 2.0, 9),
    np.zeros(9),
    np.zeros(9),
])

line_result = n3d.find_plane_normal(line_points)

print("normal:", line_result.normal)
print("planarity score:", line_result.planarity_score)
print("linearity risk:", line_result.linearity_risk)
print("eigenvalues:", line_result.eigenvalues)

normal: [0. 1. 0.]
planarity score: 1.0
linearity risk: 1.0
eigenvalues: [ 0.  0. 15.]


Here `planarity_score` is 1, but `linearity_risk` is also 1, correctly flagging that the normal can rotate arbitrarily around the line axis. The specific `normal` returned by the eigensolver should not be assigned physical meaning in this degenerate case.

The same caution applies to a fully coincident point cloud, for which every plane orientation is underdetermined.


## Downstream use in `DisclinationLine`

`DisclinationLine.act_calc_norm()` uses `find_plane_normal()` on the real-space coordinates of a disclination line. It returns the complete `PlaneNormalResult`, while also caching `result.normal` as `calc_norm` and `result.metric` as `calc_norm_metric`.

This makes the fitted average plane available without discarding the diagnostics needed to judge whether a strongly non-planar or nearly straight defect line has a meaningful average normal.


## Requirements and limitations

- The input must contain at least three finite 3D points.
- The fit is an unweighted least-squares plane through the centroid.
- The returned normal is unoriented: $\mathbf{n}$ and $-\mathbf{n}$ represent the same plane.
- A high `planarity_score` does not by itself guarantee a uniquely determined normal. Inspect `linearity_risk` as well.
- `thickness_rms` depends on the physical scale and units of the coordinates; the other two scalar diagnostics are dimensionless.
- Lower-dimensional point clouds are accepted so their degeneracy can be inspected rather than hidden, but their returned normal may not be physically meaningful.
